In [55]:
import pandas as pd
import requests
import plotly
import gradio as gr
import sklearn

print("Pandas:", pd.__version__)
print("Requests:", requests.__version__)
print("Plotly:", plotly.__version__)
print("Gradio:", gr.__version__)
print("Scikit-learn:", sklearn.__version__)

Pandas: 2.2.3
Requests: 2.32.4
Plotly: 5.24.1
Gradio: 6.29.0
Scikit-learn: 1.6.1


In [56]:
!pip -q install gradio requests pandas matplotlib

import requests
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from datetime import datetime

In [57]:

CITIES = {
    "Casablanca": (33.5731, -7.5898),
    "Rabat": (34.0209, -6.8416),
    "Fes": (34.0331, -5.0003),
    "Marrakech": (31.6295, -7.9811),
    "Tangier": (35.7595, -5.8340),
    "Agadir": (30.4278, -9.5981),
    "Meknes": (33.8935, -5.5473),
    "Oujda": (34.6814, -1.9086),
    "Tetouan": (35.5785, -5.3684),
    "Kenitra": (34.2610, -6.5802),
    "Safi": (32.2994, -9.2372),
    "El Jadida": (33.2316, -8.5007),
    "Beni Mellal": (32.3373, -6.3498),
    "Nador": (35.1681, -2.9335),
    "Laayoune": (27.1536, -13.2033),
}


In [58]:

def describe_weather(code):
    descriptions = {
        0: "Clear sky",
        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",
        45: "Fog",
        48: "Depositing rime fog",
        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Dense drizzle",
        56: "Light freezing drizzle",
        57: "Dense freezing drizzle",
        61: "Slight rain",
        63: "Moderate rain",
        65: "Heavy rain",
        66: "Light freezing rain",
        67: "Heavy freezing rain",
        71: "Slight snowfall",
        73: "Moderate snowfall",
        75: "Heavy snowfall",
        77: "Snow grains",
        80: "Slight rain showers",
        81: "Moderate rain showers",
        82: "Violent rain showers",
        85: "Slight snow showers",
        86: "Heavy snow showers",
        95: "Thunderstorm",
        96: "Thunderstorm with slight hail",
        99: "Thunderstorm with heavy hail",
    }
    return descriptions.get(int(code), "Unknown conditions")


In [59]:

def get_forecast(city):
    if city not in CITIES:
        raise ValueError("Please select a valid city.")

    latitude, longitude = CITIES[city]

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": [
            "weather_code",
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_probability_max",
            "precipitation_sum",
        ],
        "forecast_days": 7,
        "timezone": "auto",
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    daily = response.json()["daily"]

    forecast = pd.DataFrame({
        "Date": daily["time"],
        "Min Temperature (°C)": daily["temperature_2m_min"],
        "Max Temperature (°C)": daily["temperature_2m_max"],
        "Rain Probability (%)": daily["precipitation_probability_max"],
        "Expected Rainfall (mm)": daily["precipitation_sum"],
        "Weather": [
            describe_weather(code)
            for code in daily["weather_code"]
        ],
    })

    numeric_columns = [
        "Min Temperature (°C)",
        "Max Temperature (°C)",
        "Expected Rainfall (mm)",
        "Rain Probability (%)",
    ]

    for column in numeric_columns:
        forecast[column] = pd.to_numeric(
            forecast[column], errors="coerce"
        )

    forecast["Min Temperature (°C)"] = (
        forecast["Min Temperature (°C)"].round(1)
    )
    forecast["Max Temperature (°C)"] = (
        forecast["Max Temperature (°C)"].round(1)
    )
    forecast["Expected Rainfall (mm)"] = (
        forecast["Expected Rainfall (mm)"].round(1)
    )

    return forecast


In [60]:

def create_temperature_chart(forecast, city):
    fig, ax = plt.subplots(figsize=(10, 4.5))

    dates = forecast["Date"]
    max_temp = forecast["Max Temperature (°C)"]
    min_temp = forecast["Min Temperature (°C)"]

    # Yellow temperature lines
    ax.plot(
        dates,
        max_temp,
        color="#F4C542",
        marker="o",
        linewidth=3,
        markersize=7,
        label="Maximum temperature",
    )

    ax.plot(
        dates,
        min_temp,
        color="#FFB300",
        marker="o",
        linewidth=2.5,
        markersize=6,
        label="Minimum temperature",
    )

    # Fill the area between temperatures
    ax.fill_between(
        range(len(dates)),
        min_temp,
        max_temp,
        color="#F4C542",
        alpha=0.15,
    )

    ax.set_title(
        f"7-Day Temperature Forecast — {city}",
        fontsize=15,
        pad=15,
    )
    ax.set_xlabel("Date")
    ax.set_ylabel("Temperature (°C)")

    ax.set_xticks(range(len(dates)))
    ax.set_xticklabels(dates, rotation=35, ha="right")

    ax.grid(True, linestyle="--", alpha=0.2)
    ax.legend(frameon=False)
    ax.set_facecolor("#FFFDF5")
    fig.patch.set_facecolor("#FFFDF5")

    for spine in ax.spines.values():
        spine.set_visible(False)

    fig.tight_layout()
    return fig


In [61]:

def create_rain_chart(forecast, city):
    fig, ax1 = plt.subplots(figsize=(10, 4.5))

    dates = forecast["Date"]
    probability = forecast["Rain Probability (%)"].fillna(0)
    rainfall = forecast["Expected Rainfall (mm)"].fillna(0)

    x = range(len(dates))

    # Blue bars = probability of rain
    ax1.bar(
        x,
        probability,
        color="#2196F3",
        alpha=0.85,
        width=0.55,
        label="Rain probability (%)",
    )

    ax1.set_title(
        f"7-Day Rain Forecast — {city}",
        fontsize=15,
        pad=15,
    )
    ax1.set_xlabel("Date")
    ax1.set_ylabel("Rain probability (%)")
    ax1.set_ylim(0, 100)
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(dates, rotation=35, ha="right")
    ax1.grid(axis="y", linestyle="--", alpha=0.2)
    ax1.set_axisbelow(True)

    # Second axis = expected rainfall in mm
    ax2 = ax1.twinx()

    ax2.plot(
        x,
        rainfall,
        color="#00BCD4",
        marker="o",
        linewidth=2.5,
        markersize=6,
        label="Expected rainfall (mm)",
    )

    ax2.set_ylabel("Expected rainfall (mm)")

    # Combine legends
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()

    ax1.legend(
        lines1 + lines2,
        labels1 + labels2,
        loc="upper right",
        frameon=False,
    )

    ax1.set_facecolor("#F4FAFF")
    fig.patch.set_facecolor("#F4FAFF")

    for spine in ax1.spines.values():
        spine.set_visible(False)

    for spine in ax2.spines.values():
        spine.set_visible(False)

    fig.tight_layout()
    return fig


In [62]:

def update_dashboard(city):
    try:
        forecast = get_forecast(city)

        temperature_chart = create_temperature_chart(
            forecast, city
        )

        rain_chart = create_rain_chart(
            forecast, city
        )

        today = forecast.iloc[0]
        tomorrow = forecast.iloc[1]

        def format_value(value, suffix=""):
            if pd.isna(value):
                return "Not available"
            return f"{value:g}{suffix}"

        summary = f"""
## Weather in {city}

**Today ({today['Date']})**

- Conditions: {today['Weather']}
- Minimum temperature: {format_value(today['Min Temperature (°C)'], ' °C')}
- Maximum temperature: {format_value(today['Max Temperature (°C)'], ' °C')}
- Rain probability: {format_value(today['Rain Probability (%)'], '%')}
- Expected rainfall: {format_value(today['Expected Rainfall (mm)'], ' mm')}

**Tomorrow ({tomorrow['Date']})**

- Conditions: {tomorrow['Weather']}
- Minimum temperature: {format_value(tomorrow['Min Temperature (°C)'], ' °C')}
- Maximum temperature: {format_value(tomorrow['Max Temperature (°C)'], ' °C')}
- Rain probability: {format_value(tomorrow['Rain Probability (%)'], '%')}
- Expected rainfall: {format_value(tomorrow['Expected Rainfall (mm)'], ' mm')}
"""

        return (
            summary,
            forecast,
            temperature_chart,
            rain_chart,
        )

    except Exception as error:
        message = (
            "Could not retrieve the forecast. "
            "Check your internet connection and try again.\n\n"
            f"Details: {error}"
        )
        return message, pd.DataFrame(), None, None


In [63]:


weather_css = """
.gradio-container {
    background: linear-gradient(180deg, #eaf5ff 0%, #f8fbff 100%);
    color: #123b5d;
    font-family: Arial, sans-serif;
}

#weather-header {
    text-align: center;
    padding: 20px;
    color: #125a96;
}

#weather-header h1 {
    font-size: 34px;
    font-weight: 800;
}

.weather-card {
    background: white;
    border: 1px solid #d9eaff;
    border-radius: 18px;
    padding: 18px;
    box-shadow: 0 5px 18px rgba(35, 111, 180, 0.08);
}

button.primary {
    background: #1976d2 !important;
    color: white !important;
    border-radius: 12px !important;
    border: none !important;
}

button.primary:hover {
    background: #125a96 !important;
}

.block, .form {
    border-radius: 16px !important;
}

label, .prose {
    color: #174c76 !important;
}
```css
/* Clean blue and white weather dashboard */
.gradio-container {
    background: linear-gradient(180deg, #eaf6ff 0%, #ffffff 100%);
    color: #173b57;
}

/* Remove unwanted blue backgrounds from text */
#weather-header,
#weather-header h1,
.prose,
.prose p,
.prose h1,
.prose h2,
.prose h3,
.prose strong,
label {
    background: transparent !important;
    color: #174c76 !important;
}

/* Weather information cards */
.weather-card {
    background: #ffffff !important;
    border: 1px solid #d6eaff;
    border-radius: 16px;
    padding: 18px;
    box-shadow: 0 4px 15px rgba(30, 100, 170, 0.08);
}

/* Blue headings */
.prose h1,
.prose h2,
.prose h3 {
    color: #0878c9 !important;
}

/* Normal text selection */
::selection {
    background: #b9e0ff;
    color: #123b5d;
}
.gradio-container,
.gradio-container .prose,
.gradio-container .prose *,
.gradio-container .markdown-body,
.gradio-container .markdown-body * {
    color: #174c76 !important;
}

/* Main headings */
#weather-header,
#weather-header *,
.gradio-container h1,
.gradio-container h2,
.gradio-container h3 {
    color: #0878c9 !important;
}

/* Weather details: soft blue background */
.gradio-container .prose p {
    background: #e0f2ff !important;
    padding: 5px 10px;
    border-radius: 6px;
    margin: 6px 0;
}

/* Forecast section headings */
.gradio-container .prose strong {
    color: #075985 !important;
    background: transparent !important;
}

/* Tables */
.gradio-container table,
.gradio-container th,
.gradio-container td {
    color: #174c76 !important;
    background-color: #ffffff !important;
    border-color: #c8e3f8 !important;
}
```


"""

weather_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="sky",
    neutral_hue="slate",
    radius_size="lg",
)

with gr.Blocks(
    title="Météo Maroc",
    theme=weather_theme,
    css=weather_css
) as demo:

    gr.Markdown(
   """
    # ☀️ Météo Maroc
    ### Your Moroccan weather dashboard
    Select a city to explore temperature and rain forecasts.
    """,
    elem_id="weather-header")

    city_selector = gr.Dropdown(
        choices=list(CITIES.keys()),
        value="Rabat",
        label="Select a city",
    )

    refresh_button = gr.Button(
        "Get Weather Forecast",
        variant="primary",
    )

    summary_output = gr.Markdown()

    forecast_table = gr.Dataframe(
        label="7-Day Weather Forecast",
        interactive=False,
        wrap=True,
    )

    with gr.Tab("Temperature"):
        temperature_output = gr.Plot()

    with gr.Tab("Rain"):
        rain_output = gr.Plot()

    outputs = [
        summary_output,
        forecast_table,
        temperature_output,
        rain_output,
    ]

    refresh_button.click(
        fn=update_dashboard,
        inputs=city_selector,
        outputs=outputs,
    )

    demo.load(
        fn=update_dashboard,
        inputs=city_selector,
        outputs=outputs,
    )


/tmp/ipykernel_5239/1013953041.py:138: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


In [64]:

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://def5cb9f7d45d61f65.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
